In [1]:
!pip install -q "langchain==1.4.6" "langchain-community==0.4.2" \
                "langchain-huggingface==1.2.3" "langchain-classic==1.0.8" \
                "langchain-text-splitters==1.1.3" \
                "transformers<5" sentence-transformers datasets faiss-cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
import json
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name        = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
raw = loader.load()
print("rows loaded      :", len(raw))

# 1. undo the loader's json.dumps - NOT optional, see the warning below
def unmangle(text):
    try:
        return json.loads(text)
    except Exception:
        return text

for d in raw:
    d.page_content = unmangle(d.page_content)

# 2. drop the empty ones - this only works AFTER step 1
usable = [d for d in raw if d.page_content.strip()]
print("with real content:", len(usable))

data = usable[:500]                                      # 3. cap it
print("using            :", len(data))
print()
print(repr(data[0].page_content[:200]))   # repr, so you can SEE the escaping is gone

/tmp/ipykernel_1548/228225788.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import HuggingFaceDatasetLoader
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

rows loaded      : 15011
with real content: 4467
using            : 500

'Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 Au'


In [2]:
from collections import Counter

print("categories WITH a passage (these became your index):")
for cat, n in Counter(d.metadata["category"] for d in usable).most_common():
    print(f"   {cat:22} {n:6,}")

print("\ncategories with NO passage (these were dropped):")
for cat, n in Counter(d.metadata["category"] for d in raw
                      if not d.page_content.strip()).most_common():
    print(f"   {cat:22} {n:6,}")

print("\n--- one row in full, all four fields ---")
d = usable[0]
print("category   :", d.metadata["category"])
print("instruction:", d.metadata["instruction"])
print("context    :", d.page_content[:160], "...")
print("response   :", str(d.metadata["response"])[:160], "...")

categories WITH a passage (these became your index):
   closed_qa               1,773
   information_extraction  1,506
   summarization           1,188

categories with NO passage (these were dropped):
   open_qa                 3,742
   general_qa              2,191
   classification          2,136
   brainstorming           1,766
   creative_writing          709

--- one row in full, all four fields ---
category   : closed_qa
instruction: When did Virgin Australia start operating?
context    : Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virg ...
response   : Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. ...


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

print(len(data), "documents ->", len(docs), "chunks")
print(docs[0].page_content[:200])

500 documents -> 902 chunks
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 Au


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name    = "sentence-transformers/all-MiniLM-l6-v2",
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True},   # official text says False - see below
)

v = embeddings.embed_query("This is a test document.")
print("dimensions:", len(v))
print("first three:", [round(x, 4) for x in v[:3]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

dimensions: 384
first three: [-0.0383, 0.1235, -0.0286]


In [5]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)
print("indexed:", db.index.ntotal, "vectors")

# sanity check: does retrieval work at all, before you attach a model?
for d in db.similarity_search("What is cheesemaking?", k=2):
    print("-", d.page_content[:120].replace("\n", " "), "...")

indexed: 902 vectors
- A grilled cheese sandwich is made by placing a cheese filling, often cheddar or American cheese, between two slices of b ...
- Wine is an alcoholic drink typically made from fermented grapes. Yeast consumes the sugar in the grapes and converts it  ...


In [6]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

gen = pipeline(
    "text2text-generation",          # a task LangChain accepts - see the warning
    model="google/flan-t5-base",
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen)

print(llm.invoke("Answer in one sentence: what is cheese made from?"))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


cheese is a type of cheese that is made from a variety of different types of cheeses.


In [7]:
from langchain_classic.chains import RetrievalQA

retriever = db.as_retriever(search_kwargs={"k": 4})

qa = RetrievalQA.from_chain_type(
    llm       = llm,
    chain_type= "stuff",                 # official says "refine" - see below
    retriever = retriever,
    return_source_documents=True,        # official says False - you want True
)
print("chain ready")

chain ready


In [8]:
question = "What is cheesemaking?"

result = qa.invoke({"query": question})      # official uses .run() - deprecated

print("Q:", question)
print("A:", result["result"])

Token indices sequence length is longer than the specified maximum sequence length for this model (675 > 512). Running this sequence through the model will result in indexing errors


Q: What is cheesemaking?
A: A grilled cheese sandwich is made by placing a cheese filling, often cheddar or American cheese, between two slices of bread, which is then heated until the bread browns and the cheese melts.


In [9]:
result = qa.invoke({"query": question})

print("Q:", question)
print("A:", result["result"])
print("\nRetrieved", len(result["source_documents"]), "chunks:\n")
for i, d in enumerate(result["source_documents"], 1):
    print(f"[{i}] {d.page_content[:220].strip()}...\n")

# crude grounding check: how much of the answer appears in what was retrieved?
retrieved = " ".join(d.page_content.lower() for d in result["source_documents"])
words = [w for w in result["result"].lower().split() if len(w) > 3]
hits  = sum(w in retrieved for w in words)
print(f"grounded: {hits}/{len(words)} words appear in the retrieved text")

Q: What is cheesemaking?
A: A grilled cheese sandwich is made by placing a cheese filling, often cheddar or American cheese, between two slices of bread, which is then heated until the bread browns and the cheese melts.

Retrieved 4 chunks:

[1] A grilled cheese sandwich is made by placing a cheese filling, often cheddar or American cheese, between two slices of bread, which is then heated until the bread browns and the cheese melts. A layer of butter or mayonna...

[2] Wine is an alcoholic drink typically made from fermented grapes. Yeast consumes the sugar in the grapes and converts it to ethanol and carbon dioxide, releasing heat in the process. Different varieties of grapes and stra...

[3] The addition of pineapple to the traditional mix of tomato sauce and cheese, along with either ham or bacon, later became popular locally and eventually became a staple offering of pizzerias on a global scale. The name o...

[4] The division of labour is the separation of the tasks in any econom

In [10]:
for q in [
    "What is cheesemaking?",                      # should be in the corpus
    "What is the capital city of Uruguay?",       # almost certainly is not
    "Who won the 2026 World Cup?",                # cannot be - dolly predates it
]:
    r = qa.invoke({"query": q})
    print("Q:", q)
    print("A:", r["result"])
    print("   top chunk:", r["source_documents"][0].page_content[:90].strip(), "...\n")

Q: What is cheesemaking?
A: A grilled cheese sandwich is made by placing a cheese filling, often cheddar or American cheese, between two slices of bread, which is then heated until the bread browns and the cheese melts.
   top chunk: A grilled cheese sandwich is made by placing a cheese filling, often cheddar or American c ...

Q: What is the capital city of Uruguay?
A: unanswerable
   top chunk: São Paulo is the most populous city in Brazil, and is the capital of the state of São Paul ...

Q: Who won the 2026 World Cup?
A: Ramón Daz
   top chunk: Pep Guardiola and Carlo Ancelotti are the only managers to have won three FIFA Club World ...

